# Day 11 — ILT 1: DLQ, Replay Strategy and Operational Readiness (Runbooks, SLAs)

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Calendar slot** | Day 11, 10:30 AM – 12:00 PM |
| **Duration** | 90 minutes |
| **Mode** | Instructor-Led Training — **includes a live coded demo, not just concepts** |
| **Builds on** | Day 5 (DQ scan + quarantine), Day 9 (control tables, incremental idempotency), Day 10 (MERGE-based upsert) |
| **Real table referenced (read-only)** | `gbmart.bronze.customers` |
| **Practice tables (this session writes here only)** | `main.YOUR_SCHEMA.customers_processed`, `main.YOUR_SCHEMA.customers_dlq` |

### Learning Objectives
- Explain what a Dead-Letter Queue (DLQ) is and why it's the *general* operational pattern for "this row failed processing," independent of the reason — as distinct from Day 5's quarantine, which was one specific DQ scan's rejects
- Build a real Bronze → processed/DLQ split against a sampled `gbmart.bronze.customers` batch, tagging every failed row with a reason and a timestamp
- Write and run an idempotent **replay** function — reprocess DLQ rows, upsert them by key, and drain them from the queue, without ever double-counting
- Write and run a print-only **SLA check** — compare a simulated "last successful run" time against a freshness target, for both a CDC-style source and a batch-style source
- Read and adapt a **runbook** template for a real GlobalMart pipeline failure

---
**A note on where this session writes data:** every write below targets your own personal practice schema (`main.YOUR_SCHEMA`), never `gbmart.silver` or `gbmart.gold`. The only contact with the shared `gbmart` catalog is a **read-only** sample pulled from `gbmart.bronze.customers` — the same safety posture as Day 9/Day 10's `SHALLOW CLONE`/practice-schema approach, just simpler here, since a DLQ split doesn't need a cloned table with its own transaction history — only somewhere safe to land the results.

## Part 1 — What a DLQ Is, and Why

Every pipeline eventually meets a row it cannot process: malformed JSON from an API, a schema mismatch after an upstream change, a business-rule violation, a null in a field a downstream step assumes is always populated. There are three ways to handle that row, in ascending order of how much a production team should tolerate them:

| Approach | What happens | Why it's a problem |
|---|---|---|
| **Crash the batch** | One bad row raises an exception; the whole job fails | Every other perfectly good row in the batch is blocked too — one junk record can take down an entire nightly load |
| **Silently drop it** | The bad row is filtered out and never written anywhere | It disappears with no record it ever existed — no way to know it happened, count it, inspect it, or fix it later |
| **Route it to a DLQ** | The row is written, as-is, to a separate table — tagged with *why* it failed and *when* | The good rows keep flowing; the bad row is safe, inspectable, and replayable once the cause is understood |

A **Dead-Letter Queue (DLQ)** is that third option: a Delta table that exists purely to catch anything that fails processing, for any reason, anywhere in the pipeline. It's called a "queue" because the intent is to *drain* it — every row that lands there is expected to eventually be replayed (once fixed) or explicitly written off, not to live there forever.

### DLQ vs. Day 5's Quarantine Table — Same Mechanism, Different Scope

You already built something structurally identical to a DLQ in Day 5: `gbmart.silver.customers_quarantine`. The write pattern — split clean vs. rejected, tag the rejected rows, write both — is the same. What's different is the *scope* of what triggers it.

| | Day 5's Quarantine | This Session's DLQ |
|---|---|---|
| **Trigger** | Rows failing a specific **DQ scan's** rule set (`_dq_issue`: null customer id, invalid email, under 18, …) | **Anything** that fails **any** processing step, anywhere — a DQ rule, a transformation crash, a schema mismatch, a business-rule violation |
| **Framing** | "Is this row valid data?" — a data-quality question | "Did this row survive processing?" — an operational question, regardless of *why* it didn't |
| **Where it plugs in** | One specific Bronze → Silver transformation, once | Any pipeline stage — ingestion, transformation, a Gold aggregation step, an API call — reused everywhere |
| **What happens next** | Rows mostly stay quarantined for human review; not built to be reprocessed automatically | Explicitly designed to be **replayed** — the whole point is getting rows back out and into the good table |

In other words: quarantine is a DLQ used for exactly one purpose (a DQ scan). A DLQ is the general pattern quarantine is one instance of. This session builds the general version, plus the piece Day 5 didn't need: a replay function that safely gets rows back out again.

## Part 2 — Replay Strategy: Getting Rows Back Out Safely

A DLQ that never drains is just an expensive way to lose data slowly. Once a DLQ'd row is understood — the upstream bug is fixed, or a default/business rule has been decided for the bad value — you need to get it back into the real table. The dangerous part is *how*.

### The double-processing trap

If replay just re-runs the transformation and **appends** the result, and replay accidentally runs twice (or retries after a partial failure), the row gets written twice. Now a `fact_sales`-style downstream aggregate is wrong, and nobody notices until someone asks why revenue doubled for one customer.

### The fix: key-based MERGE, not blind append

This is the exact idempotency principle Day 9/Day 10 already established for incremental loading — replay is really just another incremental load, sourced from the DLQ table instead of a change feed:

| Step | What it does | Why it's safe to re-run |
|---|---|---|
| 1 | Read all rows currently sitting in the DLQ table | Reading never has side effects |
| 2 | Apply the fix (default value, corrected business rule, whatever was decided) | Deterministic — same input, same output, every time |
| 3 | `MERGE` the fixed rows into the processed table **by natural key** (`whenMatchedUpdate` / `whenNotMatchedInsert`) | Re-running the same `MERGE` with the same key just re-applies the same update — it can't create a second row |
| 4 | Remove the replayed rows from the DLQ table (a `MERGE ... WHEN MATCHED THEN DELETE`, keyed the same way) | Once a row is gone from the DLQ, a second replay call simply finds nothing left to do for it |

Step 3 is what makes replay safe; step 4 is what makes it *idempotent end-to-end* — call the replay function once, or ten times in a row, and the outcome is identical to calling it once. You'll prove exactly this below by calling `replay_dlq()` twice, back-to-back.

**A DLQ row is not always auto-replayable.** Some failures need a human decision first (is this actually bad data, or a business rule that needs to be written?), and some can't be fixed at all (a row with no usable key can't be written back anywhere meaningfully). A mature replay strategy distinguishes "auto-fixable, safe to replay on a schedule" from "needs a human to look at it" — don't build a replay function that blindly "fixes" everything just because it happens to be sitting in the DLQ.

## Part 3 — Runbooks: Turning Tribal Knowledge Into a Checklist

A runbook is a written, step-by-step "if this breaks, do this" document for a specific pipeline failure. The goal: whoever is on call — not necessarily the person who built the pipeline — can follow it without reconstructing the pipeline's design from memory under pressure.

Below is a real runbook for GlobalMart's actual CDC pipeline, `orders_data_ingestion_cdc` (Lakeflow Connect, query-based on `updated_at` — Day 2), written the way you'd actually keep it: a markdown checklist, versioned next to the pipeline code.

> ### Runbook: `orders_data_ingestion_cdc` failed / didn't run overnight
>
> **Symptom:** morning check shows `gbmart.bronze.orders` / `order_items` row counts haven't moved, or the Lakeflow Connect pipeline shows a failed/skipped run.
>
> **1. Detect**
> - [ ] Check the pipeline's run history in the Databricks Jobs/Pipelines UI — status, last successful run timestamp, error message if failed
> - [ ] Run the Day 9 HOL 2-style verification: current row counts, `MAX(updated_at)`, `DESCRIBE HISTORY` on `gbmart.bronze.orders`
>
> **2. Triage — likely root causes, cheapest checks first**
> - [ ] **Credential/auth expiry** — has the Postgres/Supabase connection or the storage credential (`ecomprojectscredentials`) token expired or been rotated?
> - [ ] **Source connectivity** — is the Supabase/Postgres instance reachable at all (not just from this pipeline)?
> - [ ] **Schema drift** — did a column get added/renamed/re-typed in source `globalmart.orders`/`order_items` since the last successful run? (Day 3's schema-evolution topic)
> - [ ] **Cursor column issue** — any nulls or clock-skew in `updated_at` that could stall or skip the query-based cursor?
> - [ ] **Compute/warehouse issue** — did the underlying SQL warehouse/cluster fail to start, or hit a timeout?
>
> **3. Fix**
> - [ ] Apply the fix for whichever root cause step 2 identified (refresh a credential, correct a schema mapping, patch a bad cursor value, etc.)
> - [ ] Manually trigger a re-run from the Databricks UI once the fix is in place — a manual click in the real workspace, not something this notebook automates
>
> **4. Confirm recovery**
> - [ ] Re-run the same verification checks from step 1 — row counts should now reflect the catch-up, `MAX(updated_at)` should be current
> - [ ] Check this session's SLA function against the pipeline's real last-write timestamp — should now show **MET**
> - [ ] Check whether any rows need **replay** as a side effect of the outage (e.g., a downstream job DLQ'd rows because Bronze was stale)
>
> **5. Close out**
> - [ ] Log the incident: what broke, root cause, fix applied, how long it was down
> - [ ] If this was a *new* failure mode, add it to step 2's checklist — a runbook that doesn't grow after every incident stops being useful

**The pattern generalizes.** Every pipeline in this course — the ADLS Autoloader sources, the Silver MERGE jobs, a Gold refresh — deserves the same shape of runbook: symptom, detect, triage (cheapest checks first), fix, confirm, close out. Write the shell once, then fill in pipeline-specific checks for step 2.

## Part 4 — SLAs: Defining and Measuring "Fresh Enough"

A pipeline SLA (Service Level Agreement) here means one thing: **how stale is too stale?** — a target for how quickly Bronze must reflect a change made at the source. Without a stated number, "is the pipeline healthy" has no answer; "did we meet freshness for the last 24 hours" becomes a real, checkable question once a target exists.

GlobalMart's two ingestion patterns justify two different targets:

| Source | Ingestion pattern | A reasonable freshness SLA | How you'd measure it |
|---|---|---|---|
| `orders` / `order_items` (via `orders_data_ingestion_cdc`) | Query-based CDC, cursor on `updated_at` — designed to run frequently | Bronze reflects a source change within **2 hours** | `current_timestamp()` vs. `MAX(updated_at)` in Bronze, or the pipeline's own last-run timestamp |
| `customers`, `products`, `addresses`, `payments`, `payment_methods` (ADLS Autoloader) | File-drop batch — new files land roughly daily | Bronze reflects a new file within **24 hours** | `DESCRIBE HISTORY` on the Bronze table — timestamp of the most recent commit |

These numbers are a starting point for class discussion, not a figure handed down from GlobalMart itself — in a real team, the target comes from asking downstream consumers (the Gold/BI layer, in this course's case) how stale a number they can tolerate, then working backward to a pipeline schedule that can hit it.

**Freshness SLA is not the same question as "did the pipeline succeed."** A pipeline can run green (no errors, exit code 0) and still breach its freshness SLA — e.g. no new file arrived at all, so there was nothing to fail on, but the data is now a day older than the SLA allows. Measuring freshness directly (as below) catches that; watching only for job failures does not.

The function built below does exactly this measurement — compares a last-successful-run time against a target and prints **MET** or **BREACHED**. It is intentionally **print/log only**: no email, Slack webhook, or PagerDuty call. Wiring a real alert on top of this check is an orchestration-layer concern (Databricks Workflows' own failure/notification settings) — out of scope for this session, and exactly the "don't trigger real systems" boundary this course holds everywhere.

## Part 5 — Live Coded Demo: Build, Break, Route, Replay, Check

Everything from here down is real, runnable PySpark — every cell executes against your own practice schema, reading only a small, read-only sample from `gbmart.bronze.customers`. Run each cell top to bottom; the whole notebook is safe to **Run All** as many times as you like.

### Setup — Practice Schema + Practice Tables

No `SHALLOW CLONE` needed this time — the source is read once per run, read-only, so there's no shared transaction history to isolate. The only state this session owns is the two output tables below, reset fresh on every run.

In [ ]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime, timedelta

# PhoneNumber is BIGINT in Bronze (same as Day 5) -- disable ANSI mode so a
# string cast/comparison on it never raises instead of just returning a value.
spark.conf.set("spark.sql.ansi.enabled", "false")

# ─── Personal practice schema — never write directly to shared gbmart tables ───
PRACTICE_SCHEMA = "main.YOUR_SCHEMA"   # ← replace with a schema you own
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {PRACTICE_SCHEMA}")

SOURCE_TABLE    = "gbmart.bronze.customers"        # read-only, never written to
PROCESSED_TABLE = f"{PRACTICE_SCHEMA}.customers_processed"
DLQ_TABLE       = f"{PRACTICE_SCHEMA}.customers_dlq"

# CREATE OR REPLACE (not IF NOT EXISTS): resets both tables to empty, with an
# explicit schema, every time this cell runs. That is what makes "Run All"
# twice behave identically instead of doubling the row counts on rerun.
spark.sql(f"""
    CREATE OR REPLACE TABLE {PROCESSED_TABLE} (
        customer_id             STRING,
        email                   STRING,
        email_domain            STRING,
        phone_e164              STRING,
        source                  STRING,   -- 'direct' or 'replay'
        original_failure_reason STRING,   -- NULL for 'direct' rows
        processed_at            TIMESTAMP
    ) USING DELTA
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE {DLQ_TABLE} (
        customer_id    STRING,
        email          STRING,
        phone_number   STRING,
        failure_reason STRING,
        failed_at      TIMESTAMP
    ) USING DELTA
""")

print(f"Practice tables ready:\n  {PROCESSED_TABLE}\n  {DLQ_TABLE}")

### Step 1 — Read a Small, Read-Only Sample

A stable `ORDER BY` + `LIMIT` (rather than a random sample) means every run of this notebook pulls the same 30 rows — that determinism matters for a repeatable classroom demo.

In [ ]:
SAMPLE_SIZE = 30

sample_df = (
    spark.table(SOURCE_TABLE)
        .select("CustomerID", "Email", "PhoneNumber")
        .orderBy("CustomerID")                                          # stable order every run
        .limit(SAMPLE_SIZE)
        .withColumn("PhoneNumber", col("PhoneNumber").cast("string"))   # BIGINT -> string, same as Day 5
)

print(f"Sampled {sample_df.count()} rows (read-only) from {SOURCE_TABLE}")
sample_df.show(5, truncate=False)

### Step 2 — Deliberately Inject Two Processing Failures

Real Bronze data is mostly clean, so a demo that only relies on whatever happens to already be broken isn't reliably repeatable across cohorts. Instead, corrupt a couple of rows on purpose — **by position within the sample, not by a hardcoded `CustomerID` value**, since `LIMIT` doesn't promise which physical rows you get. That keeps the injected-failure count exact no matter which real 30 customers land in your sample.

Two synthetic failures, standing in for a hypothetical downstream "loyalty & SMS-alerts enrichment" step that needs a valid email (to derive a marketing segment from the domain) and a recognizable phone shape (to normalize to `+91-XXXXXXXXXX` for SMS):
- 2 rows get their `Email` nulled out
- 2 different rows get their `PhoneNumber` replaced with an unrecognizable shape

In [ ]:
# Positions within the *sample*, not real CustomerID values -- reproducible
# regardless of which 30 real customers happen to be in the sample.
sample_ids    = [r.CustomerID for r in sample_df.select("CustomerID").collect()]
BAD_EMAIL_IDS = sample_ids[4:6]     # 2 rows  -> simulate a null email
BAD_PHONE_IDS = sample_ids[19:21]   # 2 rows  -> simulate an unrecognized phone shape

injected_df = (
    sample_df
        .withColumn("Email",
            when(col("CustomerID").isin(BAD_EMAIL_IDS), lit(None))
            .otherwise(col("Email")))
        .withColumn("PhoneNumber",
            when(col("CustomerID").isin(BAD_PHONE_IDS), lit("12345"))
            .otherwise(col("PhoneNumber")))
)

print("Rows with injected NULL email :", BAD_EMAIL_IDS)
print("Rows with injected bad phone  :", BAD_PHONE_IDS)

### Step 3 — Classify: Can This Row Be Processed?

Same "first matching rule wins" priority-chain style as Day 5's DQ scan, but the question being asked is broader than a DQ check: *can the enrichment step run on this row at all?*

In [ ]:
EMAIL_REGEX = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'

classified_df = injected_df.withColumn(
    "_failure_reason",
    # Priority order, first match wins (same style as Day 5's _dq_issue chain):
    when(col("Email").isNull() | (~col("Email").rlike(EMAIL_REGEX)), lit("NULL_OR_INVALID_EMAIL"))
    .when(~((length(col("PhoneNumber")) == 12) & col("PhoneNumber").startswith("91")), lit("PHONE_FORMAT_UNRECOGNIZED"))
    .otherwise(lit(None))
)

success_df = classified_df.filter(col("_failure_reason").isNull())
failure_df = classified_df.filter(col("_failure_reason").isNotNull())

print(f"Can be processed now : {success_df.count()}")
print(f"Routed to DLQ        : {failure_df.count()}  (>= 4 expected -- the 2+2 we injected; "
      f"a couple more just means one of your other sampled rows already had a real formatting issue)")
failure_df.groupBy("_failure_reason").count().show()

### Step 4 — Write Both Outputs to the Practice Schema

`mode("overwrite")` here (not `append`): this is a single batch's worth of processing, and overwriting means re-running just this cell — without rerunning Setup's `CREATE OR REPLACE` — still lands on the same result instead of accumulating duplicates.

In [ ]:
# Successful rows -> processed table, with the two enrichment columns computed
processed_new_df = (
    success_df
        .withColumn("email_domain", split(col("Email"), "@").getItem(1))
        .withColumn("phone_e164", concat(lit("+91-"), col("PhoneNumber").substr(3, 10)))
        .withColumnRenamed("CustomerID", "customer_id")
        .withColumnRenamed("Email", "email")
        .withColumn("source", lit("direct"))
        .withColumn("original_failure_reason", lit(None).cast("string"))
        .withColumn("processed_at", current_timestamp())
        .select("customer_id", "email", "email_domain", "phone_e164",
                "source", "original_failure_reason", "processed_at")
)
processed_new_df.write.format("delta").mode("overwrite").saveAsTable(PROCESSED_TABLE)

# Failed rows -> DLQ table, tagged with reason + timestamp
dlq_new_df = (
    failure_df
        .withColumnRenamed("CustomerID", "customer_id")
        .withColumnRenamed("Email", "email")
        .withColumnRenamed("PhoneNumber", "phone_number")
        .withColumnRenamed("_failure_reason", "failure_reason")
        .withColumn("failed_at", current_timestamp())
        .select("customer_id", "email", "phone_number", "failure_reason", "failed_at")
)
dlq_new_df.write.format("delta").mode("overwrite").saveAsTable(DLQ_TABLE)

print(f"{PROCESSED_TABLE} : {spark.table(PROCESSED_TABLE).count()} rows")
print(f"{DLQ_TABLE}       : {spark.table(DLQ_TABLE).count()} rows")
spark.table(DLQ_TABLE).display()

### Step 5 — The Replay Function

Reads whatever is currently in the DLQ, applies a deterministic fix per `failure_reason`, **upserts by `customer_id`** into the processed table (never a blind append), then drains the replayed rows out of the DLQ via a merge-delete.

In [ ]:
def replay_dlq():
    """
    Reprocess every row currently in DLQ_TABLE:
      1. Apply the fix for its failure_reason
      2. Recompute the same derived columns the direct path computes
      3. MERGE (upsert by customer_id) into PROCESSED_TABLE
      4. Merge-delete the replayed rows out of DLQ_TABLE
    Safe to call any number of times -- once nothing is left in the DLQ,
    every subsequent call is a no-op.
    """
    dlq_df = spark.table(DLQ_TABLE)
    pending = dlq_df.count()
    print(f"DLQ rows pending replay: {pending}")
    if pending == 0:
        print("Nothing to replay.")
        return

    # Fix, keyed off the reason each row was DLQ'd for. Rows only ever fail
    # ONE reason under Step 3's priority chain, so exactly one of the two
    # branches below actually changes anything per row -- the other field
    # on that row was already valid and passes through untouched.
    fixed_df = (
        dlq_df
            .withColumn("email",
                when(col("failure_reason") == "NULL_OR_INVALID_EMAIL",
                     concat(lit("customer_"), col("customer_id"), lit("@unknown.globalmart.com")))
                .otherwise(col("email")))
            .withColumn("phone_e164",
                when(col("failure_reason") == "PHONE_FORMAT_UNRECOGNIZED", lit("+91-0000000000"))
                .otherwise(concat(lit("+91-"), col("phone_number").substr(3, 10))))
            .withColumn("email_domain", split(col("email"), "@").getItem(1))
            .withColumn("source", lit("replay"))
            .withColumn("original_failure_reason", col("failure_reason"))
            .withColumn("processed_at", current_timestamp())
            .select("customer_id", "email", "email_domain", "phone_e164",
                    "source", "original_failure_reason", "processed_at")
    )

    # Upsert by natural key -- replaying the same DLQ row twice can never
    # create a duplicate in the processed table.
    processed_tbl = DeltaTable.forName(spark, PROCESSED_TABLE)
    (processed_tbl.alias("tgt")
        .merge(fixed_df.alias("src"), "tgt.customer_id = src.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    # Drain the DLQ: delete exactly the rows just replayed, via a merge-delete
    # keyed the same way -- a second call finds none of these keys left.
    dlq_tbl = DeltaTable.forName(spark, DLQ_TABLE)
    (dlq_tbl.alias("tgt")
        .merge(fixed_df.select("customer_id").alias("src"), "tgt.customer_id = src.customer_id")
        .whenMatchedDelete()
        .execute())

    print(f"Replayed {pending} row(s) -> upserted into {PROCESSED_TABLE}, removed from {DLQ_TABLE}")

### Step 6 — Run Replay Twice, Prove It's Idempotent

First call should find the rows written in Step 4 and drain them. Second call, immediately after, should find nothing left to do.

In [ ]:
print("=== Before replay ===")
print(f"processed : {spark.table(PROCESSED_TABLE).count()}")
print(f"dlq       : {spark.table(DLQ_TABLE).count()}")
print()

replay_dlq()

print()
print("=== After replay ===")
print(f"processed : {spark.table(PROCESSED_TABLE).count()}")
print(f"dlq       : {spark.table(DLQ_TABLE).count()}")

In [ ]:
# Call it again immediately -- nothing new has failed since, so expect
# "DLQ rows pending replay: 0" / "Nothing to replay.", and both counts unchanged.
replay_dlq()

print()
print(f"processed : {spark.table(PROCESSED_TABLE).count()}  (unchanged -- proves no double-processing)")
print(f"dlq       : {spark.table(DLQ_TABLE).count()}  (expected: 0)")

### Step 7 — Verify: the Processed Table Now Shows Both Paths

Group by `source` and `original_failure_reason` — you should see the original `direct` rows, plus the replayed rows, now carrying the reason they were originally DLQ'd, kept for audit purposes.

In [ ]:
spark.table(PROCESSED_TABLE) \
    .groupBy("source", "original_failure_reason") \
    .count() \
    .orderBy("source") \
    .display()

### Step 8 — The SLA-Check Function

Pure print/log output — no email, Slack, or PagerDuty call. Compares a "last successful run" timestamp (simulated with `datetime.now()` minus an interval — the same idea as `current_timestamp()` minus an interval in Spark SQL) against a freshness target in hours.

In [ ]:
def check_sla(source_name, last_success_ts, freshness_target_hours):
    """
    last_success_ts         : datetime of the last successful load for this source
    freshness_target_hours  : SLA target, e.g. 2 means "must be <= 2h stale"
    Print/log only -- intentionally no real alerting call of any kind.
    """
    now = datetime.now()
    hours_since = (now - last_success_ts).total_seconds() / 3600
    status = "MET" if hours_since <= freshness_target_hours else "BREACHED"
    marker = "PASS" if status == "MET" else "FAIL"

    print(f"[{marker}] {source_name}")
    print(f"    last successful load : {last_success_ts}")
    print(f"    checked at           : {now}")
    print(f"    hours since success  : {hours_since:.2f}h   (target: <= {freshness_target_hours}h)")
    print(f"    SLA status           : {status}")
    return status

### Step 9 — Demonstrate: One Healthy, One Breached, Both Source Types

In [ ]:
# Scenario 1 -- CDC source (orders/order_items), healthy: last run 1h ago vs a 2h target
check_sla("orders / order_items (CDC)", datetime.now() - timedelta(hours=1), freshness_target_hours=2)
print()

# Scenario 2 -- same CDC source, BREACHED: last run 6h ago vs the same 2h target
# (e.g. the pipeline silently stalled overnight -- this is exactly what the runbook above is for)
check_sla("orders / order_items (CDC)", datetime.now() - timedelta(hours=6), freshness_target_hours=2)
print()

# Scenario 3 -- Autoloader source, healthy: last run 20h ago vs a 24h daily-batch target
check_sla("customers / products (Autoloader)", datetime.now() - timedelta(hours=20), freshness_target_hours=24)

### Bonus — Checking a Real Signal (Still Read-Only)

The scenarios above use a made-up `last_success_ts` so both **MET** and **BREACHED** are easy to demonstrate on demand. In a real check, that timestamp comes from somewhere real — here, the actual last commit to `gbmart.bronze.customers`, via the same `DESCRIBE HISTORY` read Day 9 HOL 2 used to verify the CDC pipeline. This is a metadata read only — no write, no risk.

In [ ]:
real_last_write = (
    spark.sql(f"DESCRIBE HISTORY {SOURCE_TABLE}")
        .selectExpr("max(timestamp) as last_write")
        .collect()[0]["last_write"]
)

# Note: comparing naive local timestamps here (fine for this demo) -- a real
# production check should make timezone handling explicit on both sides.
print(f"Real last commit to {SOURCE_TABLE}: {real_last_write}")
print()
check_sla(f"{SOURCE_TABLE} (Autoloader) -- REAL signal", real_last_write, freshness_target_hours=24)

## Key Takeaways

- A **DLQ** is the general-purpose safety net for "this row failed processing" — independent of *why*. Day 5's quarantine table is one specific instance of this same mechanism, scoped to a single DQ scan.
- **Route, don't crash or drop.** Capture the row, the reason, and a timestamp, so a failure is inspectable and recoverable instead of invisible.
- **Replay must be idempotent.** A key-based `MERGE` (upsert) into the processed table, plus removing the row from the DLQ, is what makes "replay twice" produce the same result as "replay once" — the same idempotency principle Day 9/Day 10 already taught for incremental loading, applied to a queue instead of a change feed.
- Not every DLQ row is auto-replayable — some need a human decision first. A replay function that blindly "fixes" everything is its own risk.
- **SLAs measure freshness, not success** — a pipeline can exit cleanly and still be stale if nothing new arrived to process. Measure the data's age directly, as `check_sla()` does here.
- **Runbooks** turn tribal on-call knowledge into a checklist anyone can follow under pressure — write them before the first incident, in the same repo as the pipeline they cover, and grow them after every real failure.

## Try It Yourself

- [ ] Run the whole notebook top to bottom, then run it again — confirm the counts come out identical both times
- [ ] Call `replay_dlq()` a third time (after it's already drained the DLQ) and confirm it prints `Nothing to replay.`
- [ ] Change one of the `timedelta(hours=...)` values in the Step 9 demo cell and watch a **MET** scenario flip to **BREACHED**
- [ ] Add a third failure rule of your own to Step 3's priority chain (e.g., flag rows where `PreferredPaymentMethodID` is null) and trace it through the DLQ table
- [ ] Using this session's runbook template, sketch (in a markdown cell or on paper) a runbook for a *different* GlobalMart pipeline you already know — the ADLS Autoloader ingestion is a good candidate